# Ablation run: `baseline`

Self-contained candidate-detection ablation notebook. The configured ablation switches are listed below.

- Switch set: `{"bonferroni": true, "modality": "fused", "slic": true, "slope": true}`
- Fixed cohort: `df.iloc[900:1000]`
- Study root: `ablations/study_01`

This notebook is fully self-contained. Edit its configuration cell directly to control this run.

## Kaggle setup

1. In **Settings -> Internet**, turn internet **on**.
2. Attach the `landslide-reproted` dataset so `landslides_from_2018_to_2026.csv` appears under `/kaggle/input/`.
3. Add the Kaggle secret `huggingface_token` with read/write access to the output dataset repository.
4. Confirm `RAW_COMMIT_SHA`, `OUTPUT_REPO`, `OUTPUT_REVISION`, `STUDY_ROOT`, and the ablation switches below.
5. Run all cells. This notebook does not read or validate the other ablation notebooks. The default `RAW_COMMIT_SHA = 'main'` is resolved once to an immutable commit at startup. For comparison, copy the printed SHA into `RAW_COMMIT_SHA` in every run notebook and use the same `STUDY_ROOT`; a branch can move between runs.
6. Reruns skip incidents with remote candidates and successful zero-candidate records, then process missing incidents. Keep the original study configuration. Do not run two copies of the *same* variant at once.
7. New candidates, checkpoints, and summaries are staged locally and published in one final commit. Only this run's local raw downloads are deleted after each incident. Keep enough disk space for accumulated candidate chips; interruption before publication requires reprocessing any unpublished incidents.

In [1]:
%pip install -q "numpy>=1.24" "pandas>=2.0" "scipy>=1.10" "rasterio>=1.3" "scikit-image>=0.19,<2" "huggingface_hub>=0.23"

Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import csv
import hashlib
import importlib.metadata
import json
import os
import platform
import random
import re
import shutil
import socket
import tempfile
import time
import traceback
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
import rasterio
from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download
from rasterio.transform import xy
from rasterio.warp import Resampling, calculate_default_transform, reproject, transform_bounds
from rasterio.windows import Window, transform as window_transform
from scipy.stats import chi2, norm
from skimage.filters import apply_hysteresis_threshold
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects
from skimage.segmentation import slic

VARIANT_NAME = 'baseline'
ABLATION_SWITCHES = {'modality': 'fused', 'slope': True, 'slic': True, 'bonferroni': True}
RAW_REPO = 'sasudo2/landslides'
RAW_COMMIT_SHA = 'main'  # Resolved once to an immutable SHA at startup.
OUTPUT_REPO = 'sasudo2/landslides'
OUTPUT_REVISION = 'main'
STUDY_ROOT = 'ablations/study_01'
SOURCE_CSV_NAME = 'landslides_from_2018_to_2026.csv'
COHORT_START = 900
COHORT_STOP = 1000
WORKERS = 4
RAW_ROOT = "raw_images/raw_incidents"
TARGET_CRS = "EPSG:4326"
STUDY_SCHEMA_VERSION = "ablation-study-v2"
OUTPUT_SCHEMA_VERSION = "ablation-output-v1"
DETECTION_ALGORITHM_VERSION = "ablation-detection-v2"
STATUS_FIELDS = ["incident_id", "status", "n_candidates", "error"]
META_FIELDS = [
    "incident_id", "candidate_id", "area_m2", "elongation", "severity", "score",
    "lon_min", "lat_min", "lon_max", "lat_max",
]
PACKAGE_NAMES = ("numpy", "pandas", "scipy", "rasterio", "scikit-image", "huggingface_hub")
MANDATORY_RAW_SUFFIXES = ("after.tif", "slope.tif", "sar_pre.tif", "sar_post.tif")
BEFORE_RAW_SUFFIXES = ("planet_before.tif", "gee_before.tif")
REQUIRED_RAW_INPUTS = {
    "mandatory": list(MANDATORY_RAW_SUFFIXES),
    "before_any_of": list(BEFORE_RAW_SUFFIXES),
}
SETTINGS = {
    "scale_m": 10.0,
    "min_slope_deg": 20.0,
    "target_superpixel_m": 30.0,
    "high_percentile": 99.7,
    "low_percentile": 93.0,
    "alpha": 0.05,
    "max_iters": 20,
    "irmad_tol": 1e-3,
    "irmad_reg": 1e-6,
    "min_blob_area_m2": 3000.0,
    "max_blob_area_m2": 2000000.0,
    "max_elongation": 5.0,
    "max_candidates": 100,
    "chip_pad_m": 300.0,
    "slic_compactness": 0.2,
    "min_mask_objects_px": 5,
    "slic_min_segments": 50,
    "m_per_deg_lat": 111320.0,
}


def resolve_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError as exc:
        raise RuntimeError("Set HF_TOKEN or run on Kaggle with the huggingface_token secret attached") from exc
    try:
        return UserSecretsClient().get_secret("huggingface_token")
    except Exception as exc:
        raise RuntimeError("Attach the Kaggle secret 'huggingface_token' with read/write access") from exc


HF_TOKEN = resolve_hf_token()
API = HfApi(token=HF_TOKEN)

In [3]:
def norm01(x):
    a = np.nanmin(x)
    b = np.nanmax(x)
    if not np.isfinite(a) or not np.isfinite(b) or b <= a:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - a) / (b - a)).astype(np.float32)


def wmean_cov(x, weights):
    weights = weights / np.sum(weights)
    mean = np.sum(x * weights[:, None], axis=0)
    centered = x - mean
    return mean, (centered * weights[:, None]).T @ centered


def cca_axes(sxx, syy, sxy, reg=None):
    if reg is None:
        reg = SETTINGS["irmad_reg"]
    lx = np.linalg.cholesky(sxx + reg * np.eye(sxx.shape[0]))
    ly = np.linalg.cholesky(syy + reg * np.eye(syy.shape[0]))
    inv_lx = np.linalg.inv(lx)
    inv_ly = np.linalg.inv(ly)
    u, _, vt = np.linalg.svd(inv_lx @ sxy @ inv_ly.T, full_matrices=False)
    return inv_lx.T @ u, inv_ly.T @ vt.T


def irmad(x, y, max_iters=None, tol=None):
    if max_iters is None:
        max_iters = SETTINGS["max_iters"]
    if tol is None:
        tol = SETTINGS["irmad_tol"]
    n, p = x.shape
    weights = np.ones(n, dtype=np.float64)
    previous = None
    chi = None
    for _ in range(max_iters):
        mx, sxx = wmean_cov(x, weights)
        my, syy = wmean_cov(y, weights)
        xc = x - mx
        yc = y - my
        sxy = (xc * (weights / np.sum(weights))[:, None]).T @ yc
        a, b = cca_axes(sxx, syy, sxy)
        mad = xc @ a[:, :p] - yc @ b[:, :p]
        std = np.std(mad, axis=0) + 1e-6
        chi = np.sum((mad / std[None, :]) ** 2, axis=1)
        no_change_p = 1.0 - chi2.cdf(chi, df=p)
        if previous is not None and np.mean(np.abs(no_change_p - previous)) < tol:
            break
        previous = no_change_p
        weights = np.clip(no_change_p, 1e-6, 1.0)
    if chi is None:
        raise ValueError("IR-MAD requires at least one iteration")
    return chi, chi2.cdf(chi, df=p)


def build_target_grid(ref_path, scale_m=None):
    if scale_m is None:
        scale_m = SETTINGS["scale_m"]
    with rasterio.open(ref_path) as src:
        _, bottom, _, top = transform_bounds(src.crs, TARGET_CRS, *src.bounds)
        mean_lat_rad = np.deg2rad((bottom + top) / 2.0)
        m_per_deg_lon = SETTINGS["m_per_deg_lat"] * max(np.cos(mean_lat_rad), 1e-6)
        transform, width, height = calculate_default_transform(
            src.crs,
            TARGET_CRS,
            src.width,
            src.height,
            *src.bounds,
            resolution=(scale_m / m_per_deg_lon, scale_m / SETTINGS["m_per_deg_lat"]),
        )
    if width is None or height is None:
        raise ValueError(f"Could not determine analysis-grid dimensions for {ref_path}")
    return transform, int(width), int(height), mean_lat_rad


def reproject_bands(path, indexes, transform, shape):
    out = np.zeros((len(indexes), *shape), dtype=np.float32)
    with rasterio.open(path) as src:
        for i, index in enumerate(indexes):
            source = src.read(index).astype(np.float32)
            source[src.read_masks(index) == 0] = np.nan
            reproject(
                source=source,
                destination=out[i],
                src_transform=src.transform,
                src_crs=src.crs,
                src_nodata=np.nan,
                dst_transform=transform,
                dst_crs=TARGET_CRS,
                dst_nodata=np.nan,
                resampling=Resampling.bilinear,
            )
            out[i][~np.isfinite(out[i])] = 0.0
    if not np.all(np.isfinite(out)):
        raise ValueError(f"Non-finite raster values in {path}")
    return out


def _looks_like_alpha_band(band):
    values = band[np.isfinite(band)]
    if values.size < 100:
        return False
    low, high = np.percentile(values, [1, 99])
    return bool(high - low < 2.0 or len(np.unique(values)) <= 8)


def select_rgb_band_indexes(path):
    with rasterio.open(path) as src:
        if src.count >= 4:
            return [1, 2, 3] if _looks_like_alpha_band(src.read(4)) else [3, 2, 1]
        if src.count >= 3:
            return [1, 2, 3]
        raise ValueError(f"Optical raster has too few bands ({src.count}) in {path}")


def load_optical(before_path, after_path, transform, shape):
    before_indexes = select_rgb_band_indexes(before_path)
    after_indexes = select_rgb_band_indexes(after_path)
    n_bands = min(len(before_indexes), len(after_indexes), 3)
    before_indexes = before_indexes[:n_bands]
    after_indexes = after_indexes[:n_bands]
    before = reproject_bands(before_path, before_indexes, transform, shape)
    after = reproject_bands(after_path, after_indexes, transform, shape)
    meta = {
        "before_rgb_band_indexes": before_indexes,
        "after_rgb_band_indexes": after_indexes,
    }
    return before, after, meta


def load_slope(path, transform, shape):
    return reproject_bands(path, [1], transform, shape)[0]


def load_sar(pre_path, post_path, transform, shape):
    return (
        reproject_bands(pre_path, [1, 2], transform, shape),
        reproject_bands(post_path, [1, 2], transform, shape),
    )


def crop_native(path, indexes, bbox):
    with rasterio.open(path) as src:
        left, bottom, right, top = transform_bounds(TARGET_CRS, src.crs, *bbox)
        win = src.window(left, bottom, right, top)
        c0 = max(0, int(np.floor(win.col_off)))
        r0 = max(0, int(np.floor(win.row_off)))
        c1 = min(src.width, int(np.ceil(win.col_off + win.width)))
        r1 = min(src.height, int(np.ceil(win.row_off + win.height)))
        if c1 <= c0 or r1 <= r0:
            raise ValueError(f"Candidate chip does not overlap {path}")
        win = Window.from_slices((r0, r1), (c0, c1))
        return src.read(indexes, window=win).astype(np.float32), src.window_transform(win), src.crs


def write_chip(path, array, transform, crs):
    data = array[None, ...] if array.ndim == 2 else array
    with rasterio.open(
        path,
        "w",
        driver="GTiff",
        dtype="float32",
        count=data.shape[0],
        height=data.shape[1],
        width=data.shape[2],
        crs=crs,
        transform=transform,
    ) as dst:
        dst.write(data.astype(np.float32))

In [4]:
class InsufficientCoverageError(ValueError):
    pass


def change_evidence(before, after, optical):
    bands, height, width = before.shape
    if optical:
        valid = np.all(before > 0, axis=0) & np.all(after > 0, axis=0)
    else:
        valid = np.all(before != 0, axis=0) & np.all(after != 0, axis=0)
    n_valid = int(np.count_nonzero(valid))
    if n_valid <= 100:
        modality = "Optical" if optical else "SAR"
        raise InsufficientCoverageError(
            f"{modality} has {n_valid} valid paired pixels; IR-MAD requires more than 100"
        )
    x = np.moveaxis(before, 0, -1)[valid]
    y = np.moveaxis(after, 0, -1)[valid]
    chi_values, confidence_values = irmad(x, y)
    if not np.all(np.isfinite(chi_values)):
        raise ValueError("IR-MAD produced non-finite change evidence")
    confidence = np.zeros((height, width), dtype=np.float32)
    chi = np.zeros((height, width), dtype=np.float64)
    df = np.zeros((height, width), dtype=np.float64)
    confidence[valid] = confidence_values
    chi[valid] = chi_values
    df[valid] = bands
    return confidence, chi, df, valid


def segmentation_features(confidence, post, slope):
    channels = [confidence[..., None]]
    channels.append(np.stack([norm01(band) for band in post], axis=-1))
    if ABLATION_SWITCHES["slope"]:
        channels.append(norm01(slope)[..., None])
    return np.concatenate(channels, axis=-1)


def aggregate_segments(confidence, chi, df, terrain, segments):
    ids, inverse = np.unique(segments, return_inverse=True)
    inverse = inverse.reshape(segments.shape)
    active = inverse[terrain]
    counts = np.bincount(active, minlength=len(ids))
    conf_sum = np.bincount(active, weights=confidence[terrain], minlength=len(ids))
    chi_sum = np.bincount(active, weights=chi[terrain], minlength=len(ids))
    df_sum = np.bincount(active, weights=df[terrain], minlength=len(ids))
    conf_mean = np.divide(conf_sum, counts, out=np.zeros(len(ids)), where=counts > 0)
    z = np.divide(chi_sum - df_sum, np.sqrt(2.0 * df_sum), out=np.zeros(len(ids)), where=df_sum > 0)
    n_tests = max(int(np.count_nonzero(counts)), 1)
    return conf_mean[inverse].astype(np.float32), z[inverse].astype(np.float32), n_tests


def detection_mask(confidence, chi, df, terrain, features, px_area):
    if ABLATION_SWITCHES["slic"]:
        pixels_per_segment = max(1, int(SETTINGS["target_superpixel_m"] ** 2 / max(px_area, 1e-6)))
        n_segments = max(SETTINGS["slic_min_segments"], int(confidence.size / pixels_per_segment))
        segments = slic(
            features,
            n_segments=n_segments,
            compactness=SETTINGS["slic_compactness"],
            start_label=1,
            channel_axis=-1,
        )
        seg_conf, z, n_tests = aggregate_segments(confidence, chi, df, terrain, segments)
    else:
        seg_conf = confidence
        z = np.divide(chi - df, np.sqrt(2.0 * df), out=np.zeros_like(chi), where=df > 0)
        n_tests = max(confidence.size, 1)
    alpha = SETTINGS["alpha"] / n_tests if ABLATION_SWITCHES["bonferroni"] else SETTINGS["alpha"]
    z_crit = norm.ppf(1.0 - alpha)
    valid_conf = seg_conf[terrain]
    if valid_conf.size == 0 or np.max(valid_conf) <= 0:
        return np.zeros_like(terrain, dtype=bool), n_tests, float(z_crit)
    high = np.percentile(valid_conf, SETTINGS["high_percentile"])
    low = min(np.percentile(valid_conf, SETTINGS["low_percentile"]), high)
    mask = apply_hysteresis_threshold(seg_conf, low, high)
    mask = mask & terrain & (z >= z_crit)
    mask = remove_small_objects(mask, min_size=SETTINGS["min_mask_objects_px"])
    return mask, n_tests, float(z_crit)


def detect(paths):
    transform, width, height, latitude = build_target_grid(paths["before"])
    shape = (height, width)
    slope = load_slope(paths["slope"], transform, shape)
    evidence = []
    post_features = None
    optical_meta = None
    if ABLATION_SWITCHES["modality"] in ("optical", "fused"):
        before, after, optical_meta = load_optical(paths["before"], paths["after"], transform, shape)
        evidence.append(change_evidence(before, after, True))
        post_features = after
    if ABLATION_SWITCHES["modality"] in ("sar", "fused"):
        pre, post = load_sar(paths["sar_pre"], paths["sar_post"], transform, shape)
        evidence.append(change_evidence(pre, post, False))
        if ABLATION_SWITCHES["modality"] == "sar":
            post_features = post
    if post_features is None:
        raise ValueError(f"Unknown modality: {ABLATION_SWITCHES['modality']}")
    valid = np.logical_and.reduce([item[3] for item in evidence])
    n_joint_valid = int(np.count_nonzero(valid))
    if ABLATION_SWITCHES["modality"] == "fused" and n_joint_valid <= 100:
        raise InsufficientCoverageError(
            f"Fused modalities have {n_joint_valid} jointly valid paired pixels; "
            "detection requires more than 100"
        )
    confidence = np.power(
        np.prod(np.stack([item[0] for item in evidence], axis=0), axis=0),
        1.0 / len(evidence),
    ).astype(np.float32)
    confidence[~valid] = 0.0
    chi = np.sum([item[1] for item in evidence], axis=0)
    chi[~valid] = 0.0
    df = np.sum([item[2] for item in evidence], axis=0)
    df[~valid] = 0.0
    if ABLATION_SWITCHES["slope"]:
        terrain = valid & (slope >= SETTINGS["min_slope_deg"])
    else:
        terrain = valid
    confidence *= terrain
    features = segmentation_features(confidence, post_features, slope) if ABLATION_SWITCHES["slic"] else None
    m_per_deg_lon = SETTINGS["m_per_deg_lat"] * max(np.cos(latitude), 1e-6)
    px_area = abs(transform.a * transform.e) * m_per_deg_lon * SETTINGS["m_per_deg_lat"]
    mask, n_tests, z_crit = detection_mask(confidence, chi, df, terrain, features, px_area)
    candidates = []
    for region in regionprops(label(mask), intensity_image=confidence):
        area = region.area * px_area
        if not SETTINGS["min_blob_area_m2"] <= area <= SETTINGS["max_blob_area_m2"]:
            continue
        elongation = (region.major_axis_length or 1) / max(region.minor_axis_length or 1, 1e-6)
        if elongation > SETTINGS["max_elongation"]:
            continue
        r0, c0, r1, c1 = region.bbox
        lon_min, lat_max = xy(transform, r0, c0, offset="ul")
        lon_max, lat_min = xy(transform, r1, c1, offset="ul")
        severity = float(region.mean_intensity)
        candidates.append({
            "area_m2": float(area),
            "elongation": float(elongation),
            "severity": severity,
            "score": float(area * severity),
            "lon_min": float(lon_min),
            "lat_min": float(lat_min),
            "lon_max": float(lon_max),
            "lat_max": float(lat_max),
            "bbox_px": [int(r0), int(c0), int(r1), int(c1)],
        })
    candidates.sort(key=lambda item: item["score"], reverse=True)
    return {
        "candidates": candidates[:SETTINGS["max_candidates"]],
        "mask": mask,
        "slope": slope,
        "transform": transform,
        "px_area": px_area,
        "n_tests": n_tests,
        "z_crit": z_crit,
        "optical_meta": optical_meta,
    }


def export_candidates(incident_id, paths, result, folder):
    height, width = result["mask"].shape
    transform = result["transform"]
    pad = max(1, int(round(SETTINGS["chip_pad_m"] / max(np.sqrt(result["px_area"]), 1e-6))))
    optical_meta = result["optical_meta"]
    if optical_meta is None:
        optical_meta = {
            "before_rgb_band_indexes": select_rgb_band_indexes(paths["before"]),
            "after_rgb_band_indexes": select_rgb_band_indexes(paths["after"]),
        }
    rows = []
    for candidate_id, candidate in enumerate(result["candidates"], 1):
        row = {key: value for key, value in candidate.items() if key != "bbox_px"}
        row.update(incident_id=incident_id, candidate_id=candidate_id)
        r0, c0, r1, c1 = candidate["bbox_px"]
        r0 = max(0, r0 - pad)
        c0 = max(0, c0 - pad)
        r1 = min(height, r1 + pad)
        c1 = min(width, c1 + pad)
        chip_tf = window_transform(Window.from_slices((r0, r1), (c0, c1)), transform)
        left, top = chip_tf * (0, 0)
        right, bottom = chip_tf * (c1 - c0, r1 - r0)
        bbox = (left, bottom, right, top)
        chip_dir = folder / f"candidate_{candidate_id}"
        chip_dir.mkdir(parents=True)
        specs = {
            "before": crop_native(paths["before"], optical_meta["before_rgb_band_indexes"], bbox),
            "after": crop_native(paths["after"], optical_meta["after_rgb_band_indexes"], bbox),
            "slope": (result["slope"][r0:r1, c0:c1], chip_tf, TARGET_CRS),
            "mask": (result["mask"][r0:r1, c0:c1], chip_tf, TARGET_CRS),
        }
        for chip_type, (array, chip_transform, chip_crs) in specs.items():
            write_chip(
                chip_dir / f"incident_{incident_id}_candidate_{candidate_id}_{chip_type}.tif",
                array,
                chip_transform,
                chip_crs,
            )
        rows.append(row)
    return rows

In [5]:
TRANSIENT_STATUS_CODES = {408, 409, 412, 425, 429, 500, 502, 503, 504}


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def write_json(path, data):
    Path(path).write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def write_csv(path, fields, rows):
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def _status_code(exc):
    response = getattr(exc, "response", None)
    return getattr(response, "status_code", None)


def is_transient_hub_error(exc):
    status = _status_code(exc)
    if status in TRANSIENT_STATUS_CODES:
        return True
    if isinstance(exc, (ConnectionError, TimeoutError)):
        return True
    message = str(exc).lower()
    markers = (
        "timeout", "timed out", "temporarily", "connection reset", "connection aborted",
        "rate limit", "too many requests", "server error", "bad gateway",
        "service unavailable", "gateway timeout",
    )
    return any(marker in message for marker in markers) or status in TRANSIENT_STATUS_CODES


def retry_hub(action, description, attempts=5, base_delay=2.0):
    for attempt in range(1, attempts + 1):
        try:
            return action()
        except Exception as exc:
            if attempt >= attempts or not is_transient_hub_error(exc):
                raise
            delay = min(base_delay ** attempt, 60.0) + random.uniform(0.0, 1.0)
            print(f"{description} failed on attempt {attempt}/{attempts}: {exc}; retrying in {delay:.1f}s")
            time.sleep(delay)
    raise RuntimeError(f"Retry loop exhausted for {description}")


def repo_files(repo, revision, prefix):
    def list_tree():
        return list(API.list_repo_tree(
            repo_id=repo,
            repo_type="dataset",
            revision=revision,
            path_in_repo=prefix,
            recursive=True,
        ))

    try:
        entries = retry_hub(list_tree, f"list files under {repo}@{revision}/{prefix}")
    except Exception as exc:
        status = _status_code(exc)
        message = str(exc).lower()
        if status == 404 or "not found" in message or "entry not found" in message:
            return set()
        raise
    files = set()
    for entry in entries:
        path = getattr(entry, "path", None)
        if path and (getattr(entry, "type", None) == "file" or hasattr(entry, "blob_id")):
            files.add(path)
    return files


def resolve_immutable_sha(repo, configured):
    if not isinstance(configured, str) or not configured.strip():
        raise ValueError("Set RAW_COMMIT_SHA to a commit SHA, branch, or tag such as 'main'")
    configured = configured.strip()
    if configured == "REPLACE_WITH_40_CHARACTER_RAW_COMMIT_SHA":
        raise ValueError("Replace the RAW_COMMIT_SHA placeholder with 'main' or a raw-data commit SHA")
    is_full_sha = re.fullmatch(r"[0-9a-fA-F]{40}", configured) is not None
    revision = configured.lower() if is_full_sha else configured
    info = retry_hub(
        lambda: API.repo_info(repo_id=repo, repo_type="dataset", revision=revision),
        f"resolve {repo}@{revision}",
    )
    resolved = str(getattr(info, "sha", "")).lower()
    if not re.fullmatch(r"[0-9a-f]{40}", resolved):
        raise ValueError(f"Raw revision {revision!r} did not resolve to a full commit SHA: {resolved!r}")
    if is_full_sha and resolved != revision:
        raise ValueError(f"Raw revision resolved to {resolved}, expected {revision}")
    print(f"Pinned raw data: {repo}@{revision} -> {resolved}")
    if not is_full_sha:
        print(
            f"For comparable runs, set RAW_COMMIT_SHA = {resolved!r} in every run notebook; "
            "a branch or tag may move between runs."
        )
    return resolved


def raw_suffixes_by_incident(files):
    pattern = re.compile(rf"^{re.escape(RAW_ROOT)}/incident_(\d+)/incident_\1_(.+)$")
    grouped = {}
    for filename in files:
        match = pattern.match(filename)
        if match:
            grouped.setdefault(int(match.group(1)), set()).add(match.group(2))
    return grouped


def verify_cohort_inputs(incident_ids, grouped):
    missing = {}
    before_by_incident = {}
    mandatory = set(MANDATORY_RAW_SUFFIXES)
    for incident_id in incident_ids:
        suffixes = grouped.get(incident_id, set())
        absent = sorted(mandatory - suffixes)
        before = next((suffix for suffix in BEFORE_RAW_SUFFIXES if suffix in suffixes), None)
        if absent or before is None:
            missing[str(incident_id)] = {
                "missing_mandatory": absent,
                "missing_before": before is None,
            }
        else:
            before_by_incident[incident_id] = before
    if missing:
        raise RuntimeError(
            "Selected incidents lack required raw inputs at the pinned commit: "
            + json.dumps(missing, sort_keys=True)
        )
    return before_by_incident


def fetch_paths(incident_id, before_by_incident):
    prefix = f"{RAW_ROOT}/incident_{incident_id}/incident_{incident_id}_"
    suffix_by_key = {
        "before": before_by_incident[incident_id],
        "after": "after.tif",
        "slope": "slope.tif",
        "sar_pre": "sar_pre.tif",
        "sar_post": "sar_post.tif",
    }
    paths = {}
    for key, suffix in suffix_by_key.items():
        paths[key] = retry_hub(
            lambda key=key, suffix=suffix: hf_hub_download(
                repo_id=RAW_REPO,
                repo_type="dataset",
                revision=study["raw_commit_sha"],
                filename=f"{prefix}{suffix}",
                token=HF_TOKEN,
                cache_dir=str(LOCAL_RAW_ROOT / f"incident_{incident_id}"),
            ),
            f"download {prefix}{suffix}",
        )
    return paths


def package_versions():
    versions = {}
    for package in PACKAGE_NAMES:
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = "missing"
    return versions


def run_environment():
    return {
        "python": platform.python_version(),
        "implementation": platform.python_implementation(),
        "platform": platform.platform(),
        "machine": platform.machine(),
        "processor": platform.processor(),
        "cpu_count": os.cpu_count(),
        "hostname": socket.gethostname(),
        "kaggle_kernel_run_type": os.environ.get("KAGGLE_KERNEL_RUN_TYPE"),
    }


def locate_source_csv():
    explicit = Path("/kaggle/input/landslide-reproted") / SOURCE_CSV_NAME
    if explicit.is_file():
        return explicit
    matches = []
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        matches.extend(sorted(input_root.rglob(SOURCE_CSV_NAME)))
    local = Path.cwd() / SOURCE_CSV_NAME
    if local.is_file():
        matches.append(local)
    if not matches:
        raise FileNotFoundError(
            f"Attach the Kaggle dataset 'landslide-reproted' containing {SOURCE_CSV_NAME}"
        )
    preferred = [path for path in matches if "reproted" in str(path).lower()]
    return preferred[0] if len(preferred) == 1 else matches[0]


def build_study(study_root, incident_ids, before_by_incident, raw_sha, versions, source_csv_path):
    return {
        "study_schema_version": STUDY_SCHEMA_VERSION,
        "output_schema_version": OUTPUT_SCHEMA_VERSION,
        "detection_algorithm_version": DETECTION_ALGORITHM_VERSION,
        "variant": VARIANT_NAME,
        "switches": dict(ABLATION_SWITCHES),
        "raw_repo": RAW_REPO,
        "raw_commit_sha": raw_sha,
        "source_csv_name": SOURCE_CSV_NAME,
        "source_csv_sha256": hashlib.sha256(Path(source_csv_path).read_bytes()).hexdigest(),
        "cohort": {
            "declaration": f"df.iloc[{COHORT_START}:{COHORT_STOP}]",
            "start": COHORT_START,
            "stop": COHORT_STOP,
            "count": len(incident_ids),
        },
        "incident_ids": list(incident_ids),
        "required_raw_inputs": dict(REQUIRED_RAW_INPUTS),
        "before_file_by_incident": {str(key): value for key, value in before_by_incident.items()},
        "settings": dict(SETTINGS),
        "package_versions": versions,
        "created_at": utc_now(),
        "run_environment": run_environment(),
        "study_root": study_root,
    }



def read_output_file(filename):
    return Path(retry_hub(
        lambda: hf_hub_download(
            repo_id=OUTPUT_REPO,
            repo_type="dataset",
            revision=output_snapshot,
            filename=filename,
            token=HF_TOKEN,
        ),
        f"read existing output {filename}",
    ))


def load_resume_state(files, prefix, incident_ids):
    def optional_csv(name):
        if prefix + name not in files:
            return []
        with read_output_file(prefix + name).open(newline="", encoding="utf-8") as handle:
            return list(csv.DictReader(handle))

    previous_status = {
        int(row["incident_id"]): row for row in optional_csv("candidate_status.csv")
    }
    previous_rows = {}
    for row in optional_csv("candidate_metadata.csv"):
        previous_rows.setdefault(int(row["incident_id"]), []).append(row)
    previous_diagnostics = {}
    if prefix + "diagnostics.json" in files:
        previous_diagnostics = {
            int(item["incident_id"]): item
            for item in json.loads(read_output_file(prefix + "diagnostics.json").read_text())
        }
    pattern = re.compile(
        rf"^{re.escape(prefix)}incident_(\d+)/candidate_(\d+)/"
        rf"incident_\1_candidate_\2_(before|after|slope|mask)\.tif$"
    )
    chips = {}
    for filename in files:
        match = pattern.fullmatch(filename)
        if match:
            incident_id, candidate_id = int(match[1]), int(match[2])
            chips.setdefault(incident_id, {}).setdefault(candidate_id, set()).add(match[3])

    records, rows_by_id, diagnostics, incomplete = {}, {}, {}, set()
    for incident_id in incident_ids:
        candidate_files = chips.get(incident_id, {})
        checkpoint = f"{prefix}incident_{incident_id}/result.json"
        record = previous_status.get(incident_id)
        rows = previous_rows.get(incident_id, [])
        diagnostic = previous_diagnostics.get(incident_id)
        if checkpoint in files:
            saved = json.loads(read_output_file(checkpoint).read_text())
            if saved["record"]["incident_id"] != incident_id:
                raise ValueError(f"Incident checkpoint ID mismatch: {checkpoint}")
            record, rows, diagnostic = saved["record"], saved["rows"], saved["diagnostic"]
        zero_complete = (
            record is not None
            and record["status"] == "ok"
            and int(record["n_candidates"]) == 0
            and not candidate_files
        )
        if not candidate_files and not zero_complete:
            continue
        n_candidates = len(candidate_files)
        metadata_ids = [int(row["candidate_id"]) for row in rows]
        complete = (
            len(metadata_ids) == len(set(metadata_ids))
            and set(metadata_ids) == set(candidate_files)
            and all(parts == {"before", "after", "slope", "mask"} for parts in candidate_files.values())
        )
        if not complete:
            incomplete.add(incident_id)
            print(
                f"WARNING: skipping incident_{incident_id}: remote candidates exist but "
                "chips or candidate metadata are incomplete. Existing files are preserved; "
                "the study will be marked incomplete."
            )
        else:
            print(f"Skipping incident_{incident_id}: {n_candidates} existing candidate(s)")
        records[incident_id] = {
            "incident_id": incident_id, "status": "ok",
            "n_candidates": n_candidates, "error": "",
        }
        rows_by_id[incident_id] = rows
        diagnostics[incident_id] = diagnostic if diagnostic is not None else {
            "incident_id": incident_id, "n_candidates": n_candidates,
            "diagnostics_available": False, "resumed": True,
        }
    return records, rows_by_id, diagnostics, incomplete


def validate_resume_study(previous, current):
    ignored = {"created_at", "run_environment"}
    conflicts = [
        key for key in current
        if key not in ignored and previous.get(key) != current[key]
    ]
    if conflicts:
        raise ValueError(
            "Cannot mix incompatible runs in this variant folder; mismatched study fields: "
            + ", ".join(conflicts)
            + ". Restore the original configuration or choose a new STUDY_ROOT."
        )


In [6]:
def validate_study_root(value):
    path = PurePosixPath(value)
    if (
        not path.parts
        or path.is_absolute()
        or value != path.as_posix()
        or any(part in (".", "..") or part.startswith(".") for part in path.parts)
        or path.parts[0] in ("candidates", "raw_images")
    ):
        raise ValueError("STUDY_ROOT must be a new relative folder such as ablations/study_01")
    return value


def initialize_variant(study_root, study):
    LOCAL_VARIANT.mkdir(parents=True, exist_ok=False)
    run = {
        "variant": VARIANT_NAME,
        "switches": dict(ABLATION_SWITCHES),
        "study_root": study_root,
        "output_repo": OUTPUT_REPO,
        "output_revision": OUTPUT_REVISION,
        "workers": WORKERS,
        "started_at": utc_now(),
        "status": "running",
        "n_resumed": len(resume_records),
    }
    write_json(LOCAL_VARIANT / "study.json", study)
    write_json(LOCAL_VARIANT / "run.json", run)
    return run


if not re.fullmatch(r"[a-z0-9_]+", VARIANT_NAME):
    raise ValueError("VARIANT_NAME must be a simple lowercase identifier")
if set(ABLATION_SWITCHES) != {"modality", "slope", "slic", "bonferroni"}:
    raise ValueError("ABLATION_SWITCHES must define modality, slope, slic, and bonferroni")
if ABLATION_SWITCHES["modality"] not in ("fused", "optical", "sar"):
    raise ValueError("ABLATION_SWITCHES['modality'] must be fused, optical, or sar")
if not all(isinstance(ABLATION_SWITCHES[key], bool) for key in ("slope", "slic", "bonferroni")):
    raise ValueError("ABLATION_SWITCHES slope, slic, and bonferroni must be booleans")
if WORKERS < 1:
    raise ValueError("WORKERS must be at least 1")
if COHORT_START != 900 or COHORT_STOP != 1000:
    raise ValueError("This study requires the fixed cohort df.iloc[900:1000]")

study_root = validate_study_root(STUDY_ROOT)
source_csv_path = locate_source_csv()
source_df = pd.read_csv(source_csv_path)
if "id" not in source_df.columns:
    raise ValueError(f"{SOURCE_CSV_NAME} has no 'id' column")
if len(source_df) < COHORT_STOP:
    raise ValueError(f"{SOURCE_CSV_NAME} has {len(source_df)} rows, need at least {COHORT_STOP}")
cohort_df = source_df.iloc[COHORT_START:COHORT_STOP]
if len(cohort_df) != COHORT_STOP - COHORT_START:
    raise ValueError("Cohort selection did not return exactly 100 rows")
incident_ids = [int(value) for value in cohort_df["id"].tolist()]
if len(set(incident_ids)) != len(incident_ids):
    raise ValueError("Cohort contains duplicate incident IDs")

output_snapshot = retry_hub(
    lambda: API.repo_info(repo_id=OUTPUT_REPO, repo_type="dataset", revision=OUTPUT_REVISION),
    "resolve output snapshot for resume",
).sha
own_prefix = f"{study_root}/{VARIANT_NAME}/"
output_files = repo_files(OUTPUT_REPO, output_snapshot, own_prefix.rstrip("/"))
previous_study = None
if own_prefix + "study.json" in output_files:
    previous_study = json.loads(read_output_file(own_prefix + "study.json").read_text())
raw_revision = RAW_COMMIT_SHA
if previous_study is not None and not re.fullmatch(r"[0-9a-fA-F]{40}", RAW_COMMIT_SHA.strip()):
    if previous_study.get("raw_repo") == RAW_REPO:
        raw_revision = previous_study["raw_commit_sha"]
        print(f"Resuming with the original raw-data commit: {raw_revision}")
raw_sha = resolve_immutable_sha(RAW_REPO, raw_revision)
raw_files = repo_files(RAW_REPO, raw_sha, RAW_ROOT)
grouped_raw = raw_suffixes_by_incident(raw_files)
before_by_incident = verify_cohort_inputs(incident_ids, grouped_raw)
versions = package_versions()
study = build_study(study_root, incident_ids, before_by_incident, raw_sha, versions, source_csv_path)
if previous_study is not None:
    validate_resume_study(previous_study, study)
    study = previous_study
elif output_files:
    study["resume_identity_verified"] = False
    print("WARNING: existing variant has no study.json; its original study identity cannot be verified.")
resume_records, resume_rows, resume_diagnostics, incomplete_incidents = load_resume_state(
    output_files, own_prefix, incident_ids
)
if not study.get("resume_identity_verified", True) and resume_records:
    incomplete_incidents.update(resume_records)
remaining_ids = [incident_id for incident_id in incident_ids if incident_id not in resume_records]
LOCAL_ROOT = Path(tempfile.mkdtemp(prefix=f"ablation_{VARIANT_NAME}_"))
LOCAL_VARIANT = LOCAL_ROOT / VARIANT_NAME
LOCAL_RAW_ROOT = LOCAL_ROOT / "raw_downloads"
RUN = initialize_variant(study_root, study)
print(
    json.dumps(
        {
            "variant": VARIANT_NAME,
            "switches": ABLATION_SWITCHES,
            "cohort_size": len(incident_ids),
            "n_resumed": len(resume_records),
            "n_remaining": len(remaining_ids),
            "raw_commit_sha": raw_sha,
            "local_work_dir": str(LOCAL_VARIANT),
        },
        sort_keys=True,
    )
)

Pinned raw data: sasudo2/landslides@main -> 8b2c2d92d7fff68dd73483b87adf314cfd10c5f7
For comparable runs, set RAW_COMMIT_SHA = '8b2c2d92d7fff68dd73483b87adf314cfd10c5f7' in every run notebook; a branch or tag may move between runs.
{"cohort_size": 100, "local_work_dir": "/tmp/ablation_baseline_1zc2zazx/baseline", "n_remaining": 100, "n_resumed": 0, "raw_commit_sha": "8b2c2d92d7fff68dd73483b87adf314cfd10c5f7", "switches": {"bonferroni": true, "modality": "fused", "slic": true, "slope": true}, "variant": "baseline"}


In [7]:
def process_incident(incident_id):
    raw_folder = LOCAL_RAW_ROOT / f"incident_{incident_id}"
    try:
        paths = fetch_paths(incident_id, before_by_incident)
        result = detect(paths)
        incident_folder = LOCAL_VARIANT / f"incident_{incident_id}"
        incident_folder.mkdir(parents=True, exist_ok=True)
        rows = export_candidates(incident_id, paths, result, incident_folder)
        diagnostic = {
            "incident_id": incident_id,
            "n_tests": result["n_tests"],
            "z_crit": result["z_crit"],
            "mask_pixels": int(np.count_nonzero(result["mask"])),
            "n_candidates": len(rows),
        }
        return rows, diagnostic, incident_folder
    finally:
        if raw_folder.exists():
            shutil.rmtree(raw_folder)


records = dict(resume_records)
rows_by_id = dict(resume_rows)
diagnostics = dict(resume_diagnostics)
insufficient_coverage_ids = set()


with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {executor.submit(process_incident, incident_id): incident_id for incident_id in remaining_ids}
    for future in as_completed(futures):
        incident_id = futures[future]
        record = {"incident_id": incident_id, "status": "ok", "n_candidates": 0, "error": ""}
        records[incident_id] = record
        try:
            rows, diagnostic, incident_folder = future.result()
        except Exception as exc:
            record.update(status="failed", error=f"{type(exc).__name__}: {exc}")
            if isinstance(exc, InsufficientCoverageError):
                insufficient_coverage_ids.add(incident_id)
            else:
                traceback.print_exc()
            shutil.rmtree(LOCAL_VARIANT / f"incident_{incident_id}", ignore_errors=True)
            print(f"{VARIANT_NAME} incident_{incident_id} failed: {exc}")
            continue
        record["n_candidates"] = len(rows)
        rows_by_id[incident_id] = rows
        diagnostics[incident_id] = diagnostic
        print(f"{VARIANT_NAME} incident_{incident_id}: {len(rows)} candidates")
        write_json(incident_folder / "result.json", {
            "record": record, "rows": rows, "diagnostic": diagnostic,
        })


records_list = [records[incident_id] for incident_id in incident_ids]
metadata_rows = []
for incident_id in incident_ids:
    metadata_rows.extend(rows_by_id.get(incident_id, []))
metadata_rows.sort(key=lambda row: (int(row["incident_id"]), int(row["candidate_id"])))
diagnostics_list = sorted(
    (diagnostics[incident_id] for incident_id in incident_ids if incident_id in diagnostics),
    key=lambda item: item["incident_id"],
)
write_csv(LOCAL_VARIANT / "candidate_status.csv", STATUS_FIELDS, records_list)
write_csv(LOCAL_VARIANT / "candidate_metadata.csv", META_FIELDS, metadata_rows)
write_json(LOCAL_VARIANT / "diagnostics.json", diagnostics_list)

n_failed = sum(record["status"] == "failed" for record in records_list)
n_ok = len(records_list) - n_failed
n_candidates = sum(record["n_candidates"] for record in records_list if record["status"] == "ok")
n_incidents_with_candidates = sum(
    record["status"] == "ok" and record["n_candidates"] > 0 for record in records_list
)
finished_at = utc_now()
summary = {
    "variant": VARIANT_NAME,
    "switches": dict(ABLATION_SWITCHES),
    "study_root": study_root,
    "insufficient_coverage_incident_ids": sorted(insufficient_coverage_ids),
    "cohort_size": len(incident_ids),
    "n_incidents": len(records_list),
    "n_ok": n_ok,
    "n_failed": n_failed,
    "n_candidates": n_candidates,
    "n_incidents_with_candidates": n_incidents_with_candidates,
    "candidates_per_successful_incident": (n_candidates / n_ok) if n_ok else 0.0,
    "complete": bool(n_failed == 0 and not incomplete_incidents and len(records_list) == len(incident_ids)),
    "metadata_complete": not incomplete_incidents,
    "incomplete_incident_ids": sorted(incomplete_incidents),
    "n_resumed": len(resume_records),
    "raw_commit_sha": study["raw_commit_sha"],
    "finished_at": finished_at,
}
write_json(LOCAL_VARIANT / "summary.json", summary)
RUN.update({
    "finished_at": finished_at,
    "status": "complete" if summary["complete"] else (
        "failed" if n_failed > len(insufficient_coverage_ids) else "incomplete"
    ),
    "insufficient_coverage_incident_ids": sorted(insufficient_coverage_ids),
    "n_incidents": len(records_list),
    "n_ok": n_ok,
    "n_failed": n_failed,
    "n_candidates": n_candidates,
    "n_incidents_with_candidates": n_incidents_with_candidates,
    "failed_incident_ids": [record["incident_id"] for record in records_list if record["status"] == "failed"],
})
write_json(LOCAL_VARIANT / "run.json", RUN)

operations = [
    CommitOperationAdd(
        path_in_repo=f"{study_root}/{VARIANT_NAME}/{path.relative_to(LOCAL_VARIANT).as_posix()}",
        path_or_fileobj=str(path),
    )
    for path in sorted(LOCAL_VARIANT.rglob("*")) if path.is_file()
]
print(f"Committing {len(operations)} staged files for {VARIANT_NAME} in one commit")
try:
    retry_hub(
        lambda: API.create_commit(
            repo_id=OUTPUT_REPO,
            repo_type="dataset",
            revision=OUTPUT_REVISION,
            operations=operations,
            commit_message=f"Publish ablation {VARIANT_NAME}: candidates and summaries",
        ),
        f"publish all staged outputs for {VARIANT_NAME}",
    )
except Exception as exc:
    raise RuntimeError(
        f"Final commit failed for {VARIANT_NAME}; no outputs are confirmed published. "
        f"Staged candidates and summaries are retained at {LOCAL_VARIANT}: {exc}"
    ) from exc

print(json.dumps(summary, sort_keys=True))
if insufficient_coverage_ids:
    print(
        f"WARNING: study incomplete: {len(insufficient_coverage_ids)} incident(s) have "
        f"insufficient paired coverage: {sorted(insufficient_coverage_ids)}. "
        "Successful outputs and failed incident details have been saved. "
        "No modality fallback or zero-detection substitution was used. "
        "Strict comparison will reject this incomplete study."
    )
    for record in records_list:
        if record["incident_id"] in insufficient_coverage_ids:
            print(f"WARNING: incident_{record['incident_id']}: {record['error']}")
other_failures = [
    record for record in records_list
    if record["status"] == "failed" and record["incident_id"] not in insufficient_coverage_ids
]
if other_failures:
    details = "; ".join(
        f"incident_{record['incident_id']}: {record['error']}" for record in other_failures
    )
    raise RuntimeError(
        f"{len(other_failures)} incident(s) failed in {VARIANT_NAME}: {details}. "
        "Inspect candidate_status.csv and rerun this notebook to retry missing incidents"
    )

raw_images/raw_incidents/incident_74971/(…): reconstructing file:   0%|          |  0.00B / 2.77MB            

raw_images/raw_incidents/incident_74962/(…): reconstructing file:   0%|          |  0.00B / 2.38MB            

raw_images/raw_incidents/incident_74970/(…): reconstructing file:   0%|          |  0.00B / 5.72MB            

raw_images/raw_incidents/incident_74971/(…): downloading bytes:           |  0.00B            

raw_images/raw_incidents/incident_74969/(…): reconstructing file:   0%|          |  0.00B / 2.77MB            

raw_images/raw_incidents/incident_74962/(…): downloading bytes:           |  0.00B            

... (1,091 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

publish all staged outputs for baseline failed on attempt 1/5: Server error '504 Gateway Time-out' for url 'https://huggingface.co/api/datasets/sasudo2/landslides/commit/main' (Amz CF ID: Yuo7OWABKXMnop4pN4zXjbPnOWPBzV7mbX4NBV3izQ_6BHl6bGuFXA==)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/504; retrying in 2.0s
{"candidates_per_successful_incident": 26.08080808080808, "cohort_size": 100, "complete": false, "finished_at": "2026-11-04T07:44:09.358864+00:00", "incomplete_incident_ids": [], "insufficient_coverage_incident_ids": [74336], "metadata_complete": true, "n_candidates": 2582, "n_failed": 1, "n_incidents": 100, "n_incidents_with_candidates": 87, "n_ok": 99, "n_resumed": 0, "raw_commit_sha": "8b2c2d92d7fff68dd73483b87adf314cfd10c5f7", "study_root": "ablations/study_01", "switches": {"bonferroni": true, "modality": "fused", "slic": true, "slope": true}, "variant": "baseline"}
